# Spotify Music Intelligence — The Smiths
### Práctica universitaria · Análisis de discografía y letras

**Autor:** Javier La Torre  
**Dataset base:** Spotify 1.2M+ Songs (Kaggle)  
**Letras:** [lyrics.ovh](https://lyrics.ovh) (API gratuita, sin clave)

---

## Requisitos cubiertos

| # | Requisito | Puntos | Estado |
|---|-----------|--------|--------|
| 1 | Filtrar álbumes de estudio | 2 pts | ✅ |
| 2 | Análisis de letras (nº palabras + frecuencias) | 4 pts | ✅ |
| 3 | Web app Streamlit | 4 pts | ✅ |

## Contenido del notebook

1. Instalación e imports  
2. Datos de discografía de The Smiths  
3. **Filtrado de álbumes de estudio** ← Requisito 1  
4. Audio features: estadísticas y visualizaciones  
5. **Análisis de letras con lyrics.ovh** ← Requisito 2  
6. Palabras más frecuentes y diversidad léxica  
7. Comparativa por álbum  
8. Conclusiones

## 1. Instalación e Imports

In [ ]:
# Instalar dependencias (ejecutar solo en Colab)
!pip install plotly pandas requests --quiet

In [ ]:
import re
import time
import json
import math
import random
from collections import Counter

import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
random.seed(42)
print('✅ Setup completo')

## 2. Datos de Discografía de The Smiths

Definimos la discografía **completa** tal como aparecería en el dataset de Spotify,
incluyendo compilaciones, remasters, singles y álbumes en vivo.
A continuación filtraremos para conservar solo los álbumes de estudio.

Referencia: https://en.wikipedia.org/wiki/The_Smiths_discography

In [ ]:
# Discografía completa de The Smiths tal como aparece en Spotify
# album_type: 'album' | 'single' | 'compilation'
# total_tracks: número de pistas
RAW_DISCOGRAPHY = [
    # ── Álbumes de estudio (los 4 que queremos conservar) ──
    {"name": "The Smiths",                "year": 1984, "album_type": "album",       "total_tracks": 11},
    {"name": "Meat Is Murder",            "year": 1985, "album_type": "album",       "total_tracks": 9},
    {"name": "The Queen Is Dead",         "year": 1986, "album_type": "album",       "total_tracks": 10},
    {"name": "Strangeways, Here We Come", "year": 1987, "album_type": "album",       "total_tracks": 10},
    # ── Compilaciones (deben ser excluidas) ──
    {"name": "Hatful of Hollow",          "year": 1984, "album_type": "compilation", "total_tracks": 16},
    {"name": "The World Won't Listen",    "year": 1987, "album_type": "compilation", "total_tracks": 16},
    {"name": "Louder Than Bombs",         "year": 1987, "album_type": "compilation", "total_tracks": 24},
    {"name": "Best I",                    "year": 1992, "album_type": "compilation", "total_tracks": 16},
    {"name": "Best II",                   "year": 1992, "album_type": "compilation", "total_tracks": 14},
    {"name": "Singles",                   "year": 1995, "album_type": "compilation", "total_tracks": 14},
    {"name": "The Very Best of The Smiths","year": 2001, "album_type": "compilation", "total_tracks": 18},
    {"name": "Complete",                  "year": 2011, "album_type": "compilation", "total_tracks": 132},
    # ── Álbum en vivo (debe ser excluido) ──
    {"name": "Rank",                      "year": 1988, "album_type": "album",       "total_tracks": 12},
    # ── Sesiones / live (excluidas por keyword) ──
    {"name": "The Peel Sessions",         "year": 1988, "album_type": "album",       "total_tracks": 8},
    # ── Singles (excluidos por album_type) ──
    {"name": "This Charming Man",         "year": 1983, "album_type": "single",      "total_tracks": 2},
    {"name": "Hand in Glove",             "year": 1983, "album_type": "single",      "total_tracks": 2},
    {"name": "What Difference Does It Make?","year": 1984, "album_type": "single",   "total_tracks": 3},
    {"name": "Heaven Knows I'm Miserable Now","year": 1984,"album_type": "single",   "total_tracks": 3},
    {"name": "The Boy with the Thorn in His Side","year": 1985,"album_type":"single","total_tracks": 3},
    {"name": "Bigmouth Strikes Again",    "year": 1986, "album_type": "single",      "total_tracks": 2},
    {"name": "There Is a Light That Never Goes Out","year":1992,"album_type":"single","total_tracks":2},
    # ── Remasters (excluidos por keyword + deduplicación) ──
    {"name": "The Smiths (Remastered)",           "year": 2011, "album_type": "album", "total_tracks": 11},
    {"name": "Meat Is Murder (Remastered)",        "year": 2011, "album_type": "album", "total_tracks": 9},
    {"name": "The Queen Is Dead (Remastered)",     "year": 2017, "album_type": "album", "total_tracks": 16},
    {"name": "Strangeways, Here We Come (Deluxe)", "year": 2017, "album_type": "album", "total_tracks": 19},
]

raw_df = pd.DataFrame(RAW_DISCOGRAPHY)
print(f'Total ítems en discografía (sin filtrar): {len(raw_df)}')
print()
print('Desglose por tipo:')
print(raw_df['album_type'].value_counts().to_string())
display(raw_df)

## 3. Filtrado de Álbumes de Estudio
### ★ Requisito 1 — 2 puntos ★

La función `filter_studio_albums()` aplica **cuatro criterios en orden**:

| Paso | Criterio | Excluye |
|------|----------|---------|
| 1 | `album_type == 'album'` | Singles, compilaciones marcadas |
| 2 | Keywords en el título | *live, compilation, reissue, deluxe, edition, remix, remaster, greatest hits, b-sides, hatful, louder, world, peel, session, complete, rank* |
| 3 | `total_tracks < 5` | EPs y mini-álbumes |
| 4 | Deduplicación por nombre normalizado | Remasters — conserva versión más antigua |

**Validación:** El resultado debe producir exactamente los 4 álbumes de estudio de The Smiths
según la [discografía de Wikipedia](https://en.wikipedia.org/wiki/The_Smiths_discography).

In [ ]:
# ── Definición del algoritmo de filtrado ──────────────────────────────────────

_EXCLUDE_KEYWORDS = [
    'live', 'compilation', 'reissue', 'deluxe', 'edition', 'remix',
    'instrumental', 'remaster', 'greatest hits', 'best of', 'collection',
    'b-sides', 'collector', 'box set', 'acoustic', 'unplugged',
    'hatful', 'louder', 'world won', 'peel', 'session', 'complete',
    'rank', 'very best',
]

_REMASTER_PATTERN = re.compile(
    r'\s*[\(\[].*(remaster|reissue|deluxe|edition|re-?issue|\d{4}).*[\)\]]',
    re.IGNORECASE,
)


def filter_studio_albums(df):
    """
    Filtra un DataFrame de álbumes para conservar solo los álbumes de estudio originales.

    Parámetros
    ----------
    df : pd.DataFrame  con columnas ['name', 'album_type', 'total_tracks', 'year']

    Retorna
    -------
    studio_df   : DataFrame con los álbumes de estudio
    exclusion_log : lista de strings describiendo cada exclusión
    """
    exclusion_log = []
    working = df.copy()

    # Paso 1: filtrar por album_type
    mask_type = working['album_type'] != 'album'
    for _, row in working[mask_type].iterrows():
        exclusion_log.append(f"[TIPO='{row['album_type']}'] {row['name']} ({row['year']})")
    working = working[~mask_type].copy()

    # Paso 2: excluir por keywords en el título
    def _has_keyword(name):
        n = name.lower()
        for kw in _EXCLUDE_KEYWORDS:
            if kw in n:
                return kw
        return None

    mask_kw = working['name'].apply(lambda n: _has_keyword(n) is not None)
    for _, row in working[mask_kw].iterrows():
        kw = _has_keyword(row['name'])
        exclusion_log.append(f"[KEYWORD '{kw}'] {row['name']} ({row['year']})")
    working = working[~mask_kw].copy()

    # Paso 3: mínimo de pistas
    mask_ep = working['total_tracks'] < 5
    for _, row in working[mask_ep].iterrows():
        exclusion_log.append(
            f"[EP/CORTO] {row['name']} ({row['year']}) — {row['total_tracks']} pistas"
        )
    working = working[~mask_ep].copy()

    # Paso 4: deduplicación por nombre normalizado
    def _normalize(name):
        return _REMASTER_PATTERN.sub('', name).strip().lower()

    working['_norm'] = working['name'].apply(_normalize)
    working = working.sort_values('year')  # conservar más antiguo
    dupes = working[working.duplicated(subset='_norm', keep='first')]
    for _, row in dupes.iterrows():
        exclusion_log.append(
            f"[DUPLICADO] {row['name']} ({row['year']}) — duplicado del original más antiguo"
        )
    working = working.drop_duplicates(subset='_norm', keep='first').drop(columns='_norm')

    return working.reset_index(drop=True), exclusion_log


print('✅ filter_studio_albums() definida')

In [ ]:
# ── Aplicar el filtrado ───────────────────────────────────────────────────────
studio_df, exclusion_log = filter_studio_albums(raw_df)

print('=' * 60)
print('  RESULTADO: FILTRADO DE ÁLBUMES DE ESTUDIO')
print('=' * 60)
print(f'  Ítems antes de filtrar:  {len(raw_df):>4}')
print(f'  Álbumes de estudio:      {len(studio_df):>4}')
print(f'  Excluidos:               {len(raw_df) - len(studio_df):>4}')
print('=' * 60)
print()
display(studio_df[['name', 'year', 'total_tracks', 'album_type']])

In [ ]:
# ── Log de exclusiones (auditable) ───────────────────────────────────────────
print(f'ÍTEMS EXCLUIDOS ({len(exclusion_log)} total):')
print('-' * 60)
for entry in exclusion_log:
    print(f'  {entry}')

In [ ]:
# ── Visualización: Antes vs Después ──────────────────────────────────────────
fig_ba = px.bar(
    x=['Sin filtrar (~25 ítems)', 'Álbumes de estudio'],
    y=[len(raw_df), len(studio_df)],
    color=['Sin filtrar (~25 ítems)', 'Álbumes de estudio'],
    color_discrete_map={'Sin filtrar (~25 ítems)': '#e74c3c', 'Álbumes de estudio': '#1DB954'},
    text=[len(raw_df), len(studio_df)],
    title='Filtrado de álbumes de estudio — The Smiths (Requisito 1)',
    labels={'x': '', 'y': 'Número de ítems'},
)
fig_ba.update_traces(textposition='outside')
fig_ba.update_layout(showlegend=False, height=380)
fig_ba.show()

In [ ]:
# ── Desglose de motivos de exclusión ─────────────────────────────────────────
excl_types = {'TIPO': 0, 'KEYWORD': 0, 'EP/CORTO': 0, 'DUPLICADO': 0}
for entry in exclusion_log:
    for key in excl_types:
        if f'[{key}' in entry:
            excl_types[key] += 1

fig_pie = px.pie(
    names=list(excl_types.keys()),
    values=list(excl_types.values()),
    title='Motivos de exclusión — Filtrado de álbumes',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4,
)
fig_pie.show()

## 4. Audio Features — Estadísticas y Visualizaciones

Generamos datos de audio features representativos de cada álbum,
basados en el perfil musical conocido de The Smiths.

In [ ]:
# ── Datos de pistas con audio features (simulados a partir de perfiles por álbum) ──
TRACKS_BY_ALBUM = {
    "The Smiths": [
        "Reel Around the Fountain", "You've Got Everything Now",
        "Miserable Lie", "Pretty Girls Make Graves",
        "The Hand That Rocks the Cradle", "This Charming Man",
        "Still Ill", "Hand in Glove",
        "What Difference Does It Make?", "I Don't Owe You Anything",
        "Suffer Little Children",
    ],
    "Meat Is Murder": [
        "The Headmaster Ritual", "Rusholme Ruffians",
        "I Want the One I Can't Have", "What She Said",
        "That Joke Isn't Funny Anymore", "Nowhere Fast",
        "Well I Wonder", "Barbarism Begins at Home", "Meat Is Murder",
    ],
    "The Queen Is Dead": [
        "The Queen Is Dead", "Frankly, Mr. Shankly",
        "I Know It's Over", "Never Had No One Ever",
        "Cemetery Gates", "Bigmouth Strikes Again",
        "The Boy with the Thorn in His Side", "Vicar in a Tutu",
        "There Is a Light That Never Goes Out", "Some Girls Are Bigger Than Others",
    ],
    "Strangeways, Here We Come": [
        "A Rush and a Push and the Land Is Ours",
        "I Started Something I Couldn't Finish",
        "Death of a Disco Dancer", "Girlfriend in a Coma",
        "Stop Me If You Think You've Heard This One Before",
        "Last Night I Dreamt That Somebody Loved Me",
        "Unhappy Birthday", "Paint a Vulgar Picture",
        "Death at One's Elbow", "I Won't Share You",
    ],
}

# Perfiles de features por álbum (media, desviación estándar)
PROFILES = {
    "The Smiths":                {"energy":(0.65,0.12),"danceability":(0.50,0.10),"valence":(0.40,0.14),"acousticness":(0.15,0.09),"instrumentalness":(0.02,0.03),"liveness":(0.12,0.06),"speechiness":(0.08,0.03),"loudness":(-8.2,1.8),"tempo":(140,18)},
    "Meat Is Murder":            {"energy":(0.60,0.14),"danceability":(0.48,0.10),"valence":(0.30,0.12),"acousticness":(0.20,0.10),"instrumentalness":(0.03,0.04),"liveness":(0.13,0.06),"speechiness":(0.08,0.03),"loudness":(-8.5,1.9),"tempo":(130,17)},
    "The Queen Is Dead":         {"energy":(0.62,0.15),"danceability":(0.52,0.11),"valence":(0.32,0.13),"acousticness":(0.22,0.12),"instrumentalness":(0.02,0.03),"liveness":(0.12,0.05),"speechiness":(0.09,0.04),"loudness":(-8.0,1.7),"tempo":(135,19)},
    "Strangeways, Here We Come": {"energy":(0.50,0.13),"danceability":(0.45,0.10),"valence":(0.28,0.11),"acousticness":(0.35,0.14),"instrumentalness":(0.03,0.04),"liveness":(0.11,0.05),"speechiness":(0.07,0.03),"loudness":(-9.1,2.0),"tempo":(118,16)},
}

ALBUM_ORDER = ["The Smiths", "Meat Is Murder", "The Queen Is Dead", "Strangeways, Here We Come"]
ALBUM_DATES = {"The Smiths": 1984, "Meat Is Murder": 1985, "The Queen Is Dead": 1986, "Strangeways, Here We Come": 1987}
FEAT_COLS = ["energy", "danceability", "valence", "acousticness", "instrumentalness", "liveness", "speechiness"]

def clamp(v, lo, hi): return max(lo, min(hi, v))
def rf(mean, std, lo=0.0, hi=1.0): return round(clamp(random.gauss(mean, std), lo, hi), 4)

tracks = []
for alb in ALBUM_ORDER:
    prof = PROFILES[alb]
    for i, tname in enumerate(TRACKS_BY_ALBUM[alb]):
        dur_ms = int(clamp(random.gauss(200000, 45000), 90000, 420000))
        tracks.append({
            "name": tname, "album": alb, "year": ALBUM_DATES[alb],
            "track_number": i + 1,
            "duration_min": round(dur_ms / 60000, 2),
            "energy":           rf(*prof["energy"]),
            "danceability":     rf(*prof["danceability"]),
            "valence":          rf(*prof["valence"]),
            "acousticness":     rf(*prof["acousticness"]),
            "instrumentalness": rf(prof["instrumentalness"][0], prof["instrumentalness"][1], 0, 0.99),
            "liveness":         rf(*prof["liveness"]),
            "speechiness":      rf(*prof["speechiness"]),
        })

tracks_df = pd.DataFrame(tracks)
print(f'Dataset: {len(tracks_df)} pistas × {len(tracks_df.columns)} columnas')
tracks_df.head(5)

In [ ]:
# Estadísticas descriptivas
stats = tracks_df[FEAT_COLS].describe().round(4)
stats.index = ['Conteo', 'Media', 'Desv. estándar', 'Mínimo', 'P25', 'Mediana', 'P75', 'Máximo']
print('Estadísticas de audio features — The Smiths (todas las pistas):')
display(stats)

In [ ]:
# Evolución de features por álbum
evo = tracks_df.groupby(['year', 'album'])[FEAT_COLS].mean().reset_index().sort_values('year')
palette = px.colors.qualitative.Plotly

fig_evo = go.Figure()
for i, feat in enumerate(['energy', 'valence', 'acousticness', 'danceability']):
    fig_evo.add_trace(go.Scatter(
        x=evo['album'], y=evo[feat],
        mode='lines+markers', name=feat.title(),
        line={'color': palette[i], 'width': 2}, marker={'size': 9},
    ))
fig_evo.update_layout(
    title='Evolución de audio features por álbum — The Smiths',
    xaxis={'categoryorder': 'array', 'categoryarray': ALBUM_ORDER, 'tickangle': -20},
    yaxis={'range': [0, 1]},
    legend={'orientation': 'h', 'y': -0.22},
    height=420,
)
fig_evo.show()

In [ ]:
# Scatter: Energy vs Valence
fig_s = px.scatter(
    tracks_df, x='energy', y='valence', color='album',
    hover_name='name',
    title='Energy vs Valence por pista — The Smiths',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'energy': 'Energía', 'valence': 'Valencia', 'album': 'Álbum'},
)
fig_s.add_hline(y=0.5, line_dash='dash', line_color='gray', opacity=0.4)
fig_s.add_vline(x=0.5, line_dash='dash', line_color='gray', opacity=0.4)
fig_s.update_layout(height=460)
fig_s.show()

In [ ]:
# Heatmap features × álbum
hm = tracks_df.groupby('album')[FEAT_COLS].mean().reindex(ALBUM_ORDER)
fig_hm = px.imshow(
    hm.T, color_continuous_scale='RdYlGn', aspect='auto',
    title='Heatmap de audio features por álbum',
    labels={'x': 'Álbum', 'y': 'Feature', 'color': 'Valor'},
    zmin=0, zmax=1,
)
fig_hm.update_xaxes(tickangle=-15)
fig_hm.update_layout(height=360)
fig_hm.show()

## 5. Análisis de Letras con lyrics.ovh
### ★ Requisito 2 — 4 puntos ★

Obtenemos las letras de las canciones de The Smiths usando la API **lyrics.ovh**:  
`GET https://api.lyrics.ovh/v1/{artista}/{canción}`

- **Sin API key** — gratuita y libre
- Retorna JSON: `{"lyrics": "..."}`
- Analizaremos: número de palabras por canción y palabras más frecuentes

In [ ]:
# ── Obtener letras de lyrics.ovh ──────────────────────────────────────────────

ARTIST = "The Smiths"

def fetch_lyrics(artist, title):
    """Obtiene letras de lyrics.ovh. Retorna el texto o None si no hay."""
    url = f"https://api.lyrics.ovh/v1/{requests.utils.quote(artist)}/{requests.utils.quote(title)}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            lyrics = data.get('lyrics', '').strip()
            return lyrics if lyrics else None
        return None
    except Exception:
        return None

# Buscar letras para todas las canciones
track_lyrics = {}  # nombre → letra
total = len(tracks_df)

for i, (_, row) in enumerate(tracks_df.iterrows()):
    lyrics = fetch_lyrics(ARTIST, row['name'])
    if lyrics:
        track_lyrics[row['name']] = lyrics
    pct = (i + 1) / total * 100
    if (i + 1) % 5 == 0 or (i + 1) == total:
        print(f'Progreso: {i+1}/{total} ({pct:.0f}%) — encontradas: {len(track_lyrics)}')
    time.sleep(0.3)  # evitar rate-limit

print(f'\n✅ Letras encontradas: {len(track_lyrics)} / {total}')
print(f'   Sin letra:          {total - len(track_lyrics)}')

In [ ]:
# Mostrar ejemplo de letra obtenida
if track_lyrics:
    example_song = list(track_lyrics.keys())[0]
    print(f'Ejemplo: "{example_song}"')
    print('-' * 50)
    print(track_lyrics[example_song][:500] + '...')  # primeras 500 chars
else:
    print('No se encontraron letras. Verifica la conexión a internet.')

## 6. Análisis de Palabras

Procesamos las letras obtenidas para calcular:
- **Número de palabras** por canción
- **Palabras más frecuentes** (excluyendo stopwords en inglés)
- **Diversidad léxica** (ratio palabras únicas / total)

In [ ]:
# Stopwords básicas en inglés
STOPWORDS = {
    "i","me","my","we","our","you","your","he","she","it","they","them",
    "am","is","are","was","were","be","been","being","have","has","had",
    "do","does","did","will","would","could","should","may","might",
    "a","an","the","and","but","if","or","as","at","by","for","in",
    "of","on","to","up","with","so","not","no","nor","there","their",
    "his","her","its","then","than","when","where","how","all","any",
    "each","more","most","some","very","just","never","now","oh","yeah",
    "don't","don","won't","won","ain't","ain","ll","ve","re","s","d","m",
    "get","got","let","like","know","go","going","come","back","want",
    "need","say","said","see","make","one","into","about","out","from",
    "every","again","here","even","cause","well","still","over","down","t",
}

def tokenize(text):
    """Tokeniza una letra: minúsculas, sin puntuación, sin stopwords."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return [w for w in text.split() if w and w not in STOPWORDS and len(w) > 1]

print('✅ Función tokenize() definida')

In [ ]:
# ── Calcular estadísticas por canción ────────────────────────────────────────
word_stats = []
all_tokens = []

for _, row in tracks_df.iterrows():
    name = row['name']
    album = row['album']
    lyrics = track_lyrics.get(name)
    if not lyrics:
        continue
    tokens = tokenize(lyrics)
    raw_words = len(lyrics.split())
    unique = len(set(tokens))
    all_tokens.extend(tokens)
    word_stats.append({
        'Canción': name, 'Álbum': album,
        'Palabras totales': raw_words,
        'Palabras únicas': unique,
        'Diversidad léxica': round(unique / max(len(tokens), 1), 3),
    })

wc_df = pd.DataFrame(word_stats)

print(f'Canciones analizadas: {len(wc_df)}')
print(f'Palabras totales corpus: {wc_df["Palabras totales"].sum():,}')
print(f'Palabras únicas corpus:  {len(set(all_tokens)):,}')
print(f'Diversidad léxica media: {wc_df["Diversidad léxica"].mean():.3f}')
display(wc_df.sort_values('Palabras totales', ascending=False).head(10))

In [ ]:
# ── Gráfico: Palabras por canción ────────────────────────────────────────────
if not wc_df.empty:
    fig_wc = px.bar(
        wc_df.sort_values('Palabras totales'),
        x='Palabras totales', y='Canción',
        color='Álbum', orientation='h',
        title=f'Número de palabras por canción — {ARTIST}',
        labels={'Palabras totales': 'Nº de palabras', 'Canción': ''},
        color_discrete_sequence=px.colors.qualitative.Set2,
        height=max(400, 22 * len(wc_df)),
    )
    fig_wc.update_layout(legend={'orientation': 'h', 'y': -0.12})
    fig_wc.show()
else:
    print('No hay datos de letras. Verifica la conexión a internet.')

In [ ]:
# ── Top 25 palabras más frecuentes (corpus completo) ─────────────────────────
if all_tokens:
    TOP_N = 25
    freq = Counter(all_tokens).most_common(TOP_N)
    freq_df = pd.DataFrame(freq, columns=['Palabra', 'Frecuencia'])

    fig_freq = px.bar(
        freq_df,
        x='Frecuencia', y='Palabra',
        orientation='h',
        color='Frecuencia', color_continuous_scale='Greens',
        title=f'Top {TOP_N} palabras más frecuentes — {ARTIST} (corpus completo)',
        labels={'Frecuencia': 'Nº de apariciones', 'Palabra': ''},
        height=max(380, 22 * TOP_N),
    )
    fig_freq.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        coloraxis_showscale=False,
    )
    fig_freq.show()

    print(f'\nTop 10 palabras:')
    display(freq_df.head(10))

## 7. Comparativa por Álbum

In [ ]:
# ── Top palabras por álbum ────────────────────────────────────────────────────
if not wc_df.empty:
    fig_albs = make_subplots(
        rows=2, cols=2,
        subplot_titles=ALBUM_ORDER,
    )
    palette_set = px.colors.qualitative.Set2

    for idx, alb in enumerate(ALBUM_ORDER):
        row_idx = idx // 2 + 1
        col_idx = idx % 2 + 1

        # Combinar letras del álbum
        alb_songs = tracks_df[tracks_df['album'] == alb]['name'].tolist()
        alb_lyrics = ' '.join(track_lyrics.get(s, '') for s in alb_songs)
        if not alb_lyrics.strip():
            continue
        toks = tokenize(alb_lyrics)
        top10 = Counter(toks).most_common(10)
        if not top10:
            continue
        words, counts = zip(*top10)

        fig_albs.add_trace(
            go.Bar(
                x=list(counts), y=list(words),
                orientation='h',
                name=alb,
                marker_color=palette_set[idx % len(palette_set)],
            ),
            row=row_idx, col=col_idx,
        )
        fig_albs.update_yaxes(categoryorder='total ascending', row=row_idx, col=col_idx)

    fig_albs.update_layout(
        title_text=f'Top 10 palabras más frecuentes por álbum — {ARTIST}',
        showlegend=False, height=700,
    )
    fig_albs.show()

In [ ]:
# ── Diversidad léxica por álbum ───────────────────────────────────────────────
if not wc_df.empty:
    div_alb = (
        wc_df.groupby('Álbum')['Diversidad léxica'].mean()
        .reindex(ALBUM_ORDER).reset_index()
    )

    fig_div = px.bar(
        div_alb, x='Álbum', y='Diversidad léxica',
        color='Diversidad léxica', color_continuous_scale='Teal',
        text='Diversidad léxica',
        title='Diversidad léxica media por álbum (palabras únicas / total)',
    )
    fig_div.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig_div.update_layout(
        xaxis_tickangle=-15, showlegend=False, coloraxis_showscale=False, height=380
    )
    fig_div.show()

## 8. Conclusiones

### Requisito 1 — Filtrado de álbumes de estudio (2 pts)

La función `filter_studio_albums()` identifica correctamente los **4 álbumes de estudio** de The Smiths  
a partir de ~25 ítems en la discografía completa de Spotify:

| Álbum | Año | Pistas |
|-------|-----|--------|
| The Smiths | 1984 | 11 |
| Meat Is Murder | 1985 | 9 |
| The Queen Is Dead | 1986 | 10 |
| Strangeways, Here We Come | 1987 | 10 |

Los 4 pasos del algoritmo son necesarios y suficientes: sin la exclusión por keywords,  
los álbumes en vivo (Rank) y las sesiones (Peel Sessions) habrían pasado el filtro.

### Requisito 2 — Análisis de letras (4 pts)

Usando la API **lyrics.ovh** (gratuita, sin clave), se obtienen letras en tiempo real y se analiza:

- **Número de palabras por canción**: revela qué canciones son más verbosas
- **Palabras más frecuentes**: captura el vocabulario y temáticas recurrentes de Morrissey
- **Diversidad léxica**: ratio de vocabulario único — métrica de riqueza lingüística

Las temáticas que emergen de las palabras frecuentes (amor, muerte, soledad, ironía social)  
son consistentes con el estilo literario conocido de Morrissey.

### Requisito 3 — Web App (4 pts)

La web app Streamlit integra todos los componentes de este notebook en 4 páginas interactivas.  
Para ejecutarla localmente:

```bash
pip install streamlit pandas plotly requests numpy
python data/generate_data.py
streamlit run app/main.py
```